# Problem 2. Two-Source Heat Localization

**Day 1 · Problem 2 · 70 points · open-ended.**
**Compute:** `optional-colab-l4`. The CPU path is required and is what the grader runs; the `Accelerator extension` section at the end is optional.

**Allowed tools.** Python's standard library, NumPy, and PyTorch only, on a CPU. No web access, downloads, external datasets, model hubs, pretrained weights, or checkpoints you did not create in the same run.

## The physical problem

A thin square plate occupies $[0,1]^2$.
At time $0$, two point sources of heat are released at unknown positions $s_1,s_2\in[0.15,0.85]^2$ with unknown strengths $q_1,q_2\in[0.5,1.5]$.
The sources are at least $0.25$ apart: $\lVert s_1-s_2\rVert\ge0.25$.
Heat then spreads by diffusion with diffusivity $D=1$ (ignore the plate's edges: treat the plate as part of an infinite sheet).

**Forward model (the Gaussian heat kernel).** The temperature rise at point $p\in\mathbb R^2$ and time $t>0$ is
$$u(p,t)=\sum_{k=1}^{2}q_k\,G_t(p-s_k),\qquad G_t(r)=\frac{1}{4\pi Dt}\exp\!\Big(-\frac{\lVert r\rVert^2}{4Dt}\Big).$$
For each $t$, $G_t$ is exactly the density of the two-dimensional Gaussian $N\big(0,\,2Dt\,I_2\big)$, so each source contributes $q_k$ times a Gaussian bump centred at $s_k$ whose variance $2Dt$ grows with time.

**Sensors and readings.** $25$ sensors sit on the grid $\{0.1,0.3,0.5,0.7,0.9\}^2$; sensor $m=5a+b$ ($a,b\in\{0,\dots,4\}$) is at $(0.1+0.2a,\;0.1+0.2b)$ (`r2_001_data.HEAT_SENSORS_XY`, shape `(25, 2)`).
Every sensor reads $u$ at two times, $t_1=0.005$ and $t_2=0.02$.
A row of inputs has $50$ readings: columns $0$–$24$ are sensors $0$–$24$ at $t_1$, columns $25$–$49$ the same sensors at $t_2$.
Each reading carries independent additive noise $N(0,0.05^2)$ and is rounded to four decimals.
`r2_001_data.heat_forward(sources_xy, strengths)` computes the noise-free readings: `sources_xy` `(N, 2, 2)` (row $k$ is source $k$'s $(x,y)$), `strengths` `(N, 2)`, output `(N, 50)`; it is differentiable.

## Task

From the $50$ readings of a row, estimate the positions of both sources.
Predictions are floating `(N, 4)` tensors $(\hat x_1,\hat y_1,\hat x_2,\hat y_2)$, in either source order.

**Metric (lower is better).** For one row, the error is the better of the two ways to pair predicted with true sources,
$$E=\min\Big\{\tfrac12\big(\lVert\hat s_1-s_1\rVert+\lVert\hat s_2-s_2\rVert\big),\;\tfrac12\big(\lVert\hat s_1-s_2\rVert+\lVert\hat s_2-s_1\rVert\big)\Big\},$$
averaged over the rows (`r2_001_data.pair_position_error_per_example`).
Strengths are not scored, but you may estimate and use them.

## Data

* `x, y = r2_001_data.train_rows("heat")`: $800$ rows, `x` float32 `(800, 50)`, `y` float32 `(800, 6)` $=(x_1,y_1,q_1,x_2,y_2,q_2)$ with the sources ordered by their $x$-coordinate.
* `r2_001_data.val_rows("heat")`: $200$ rows, same layout, for selection only.
* Locked test: $200$ rows, reachable only through `final_test_score`.
* You may also draw fresh simulated pairs with `r2_001_data.simulate("heat", n, seed)` for any `seed != 20261101`; they come from the same generative process and never coincide with validation or test rows.

## Committed baseline

`r2_001_data.baseline_score("heat")` runs per-example least squares on the 50 readings (Adam, learning rate $0.01$, $300$ steps, on positions and log-strengths) from the same fixed start for every row: sources at $(0.35,0.5)$ and $(0.65,0.5)$ with strengths $1$ (`r2_001_data.heat_ls_fit` with `heat_baseline_init`; both are importable).
It scores **B = 0.04810** at the frozen seed.

## Fixed elements (all are graded)

* **Interface and seam.** Your approach exposes `fit(train_x, train_y)` and `predict(x)`; `predict` returns the `(N, 4)` positions and is the `predict_fn` you pass to `final_test_score`. `fit` records `r2_001_data.split_of("heat", row)` for every row of `train_x` into a set `seam`; after fitting, assert that no recorded value is `"val"` or `"test"`.
* **Budget.** Create `budget = r2_001_data.StepBudget(3000)` before any fitting.
  **Every** optimizer you use, including any per-example refinement inside `predict`, is wrapped with `budget.wrap(...)`; at most 3,000 optimizer steps in total.
  An optimizer over a whole batch of examples costs one step per update, not one per example.
  Only non-iterative closed-form computation is free; any iterative solver must run as a wrapped optimizer so that every iteration counts.
  `budget.elapsed()`, measured after the test call, must be `< 60` seconds on the grading CPU.
* **Protocol.** Fit only on the rows this statement names, make every choice on `r2_001_data.val_rows("heat")`, and reach the locked test rows only through `r2_001_data.final_test_score(predict_fn, n_boot=1000, alpha=0.05, task="heat")`, called exactly once, after all fitting and selection.
  Assert `r2_001_data.test_call_count("heat") == 0` immediately before that call and `== 1` after it, and print the returned `TestScore`.
  `r2_001_data.baseline_score("heat")` may be called (it is not counted) to print the baseline under the same protocol.
* **Seam values allowed:** `{"train"}` (plus `None` for rows you simulated yourself).
* **Run clean.** The notebook runs top to bottom in a fresh kernel without errors; delete dead experiments.

## Grading

**Test score (42 points, 60%).** The grader re-executes this notebook top to bottom on a CPU under `SEED = 20261101` and reads the point estimate (`.score`) of your single `final_test_score` call.
The committed baseline scores **B = 0.04810** and the reference solution scores **R = 0.001541** at this seed; let `g = |B - R| = 0.04656`.

* **42 points** (100% of the score portion) if your score is `<= 0.01318` (that is, `<= R + 0.25 g`);
* **25.2 points** (60%) if it is `< 0.04810` (that is, `< B`);
* **10.5 points** (25%) if it is `< 0.05974` (that is, `< B + 0.25 g`);
* **0 points** otherwise.

A protocol violation (a second `final_test_score` call, a seam value outside the allowed set, the step budget or time limit exceeded, a banned tool, or any use of validation or test rows other than their stated roles) scores **0** for the test-score portion.

**Writeup (28 points, 40%).** Four parts, 7 points each, graded against the rubric descriptors in `rubric.md`:

* **Approach:** the model or method, its inputs and features, and why it suits this problem.
* **Alternatives considered:** at least two, each with the validation evidence that ruled it out.
* **Evaluation:** the validation protocol, the single final test score with its 95% bootstrap interval and `n`, and the comparison with the baseline under the same protocol.
* **Limitations:** at least one failure mode and how you would test for it.

Every number in the writeup must be traceable to an output of this notebook.

In [ ]:
import importlib.util
import json
import math
import os
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20261101
torch.set_num_threads(1)


def find_data_dir() -> Path:
    """Locate r2-001's data/ folder from the notebook's working directory."""
    candidates = [Path("../data"), Path("data"), Path("mocktests/r2-001/data"), Path("book2/mocktests/r2-001/data")]
    env_root = os.environ.get("USAAIO_BOOK_ROOT")
    if env_root:
        candidates.insert(0, Path(env_root) / "mocktests" / "r2-001" / "data")
    for path in candidates:
        if (path / "gen_r2_001.py").is_file():
            return path.resolve()
    raise FileNotFoundError("cannot find mocktests/r2-001/data; run from problems/ or set USAAIO_BOOK_ROOT")


DATA_DIR = find_data_dir()
_spec = importlib.util.spec_from_file_location("r2_001_data", DATA_DIR / "r2_001_data.py")
r2_001_data = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(r2_001_data)
assert r2_001_data.SEED == SEED

x_train, y_train = r2_001_data.train_rows("heat")
x_val, y_val = r2_001_data.val_rows("heat")
print(x_train.shape, y_train.shape, x_val.shape)
print("baseline recipe:", r2_001_data.BASELINE_RECIPES["heat"])


## Your work

Forward-model check, baseline under the protocol, validated method, seam, budget, the single test call, and answer-check assertions.

In [ ]:
# Your approach: fit, predict, validation evidence, the single test call, and assertions.


## Writeup

**Approach.** *Replace.*

**Alternatives considered.** *Replace.*

**Evaluation.** *Replace.*

**Limitations.** *Replace.*

## Accelerator extension

*Optional; Colab L4 only; never executed by the grader and never required for credit. The graded score is always the CPU run above.*

On a Colab L4, train the same kind of learned inverse on far more simulated pairs (for example `r2_001_data.simulate("heat", 500000, seed=7)`), with a wider network, mini-batches, and `bfloat16` autocast, keeping any per-example refinement in `float32`:

```python
L4_CONFIG = {"device": "cuda", "amp_dtype": torch.bfloat16, "simulated_rows": 500000, "hidden": 512,
             "steps": 20000, "batch_size": 4096, "lr": 1e-3, "refine_steps": 200, "refine_dtype": torch.float32}
```

**What to observe.** Whether the learned inverse alone, without refinement, closes the gap to the refined CPU result once it has hundreds of thousands of pairs; and where in the plate (near the centre, near the boundary, sources close together) the remaining errors concentrate.